# Genned: adversarial fine-tuning (Phase 2b, issue #15), attempt 2

Fine-tunes the app's detectors so that invisible perturbations up to **4/255** can't flip them.
Runs on a free **Colab or Kaggle GPU**.

**Why attempt 2:** attempt 1 (PGD-AT, half of each batch attacked, lr 1e-5, 2 epochs) improved clean accuracy but
left Community Forensics at **1% robust accuracy**. This run uses **TRADES** (every image gets an adversarial copy,
loss = clean BCE + 6 × KL(clean ‖ adversarial)) with a 10× higher learning rate and 8 epochs.

**Held-out data:** training uses **Defactify's train split only**. MJ/DALL·E/SD/NBP and the two video sets stay
completely unseen, so the clean-accuracy gates measure generalization, not memorized datasets.

**Before you start**
1. *Runtime → Change runtime type → GPU* (Colab), or *Accelerator → GPU T4/P100* + *Internet on* (Kaggle).
2. Secret **`HF_TOKEN`** (write scope; account has accepted the gated `Dafilab/ai-image-detector` terms).
   Colab: key icon in the left sidebar (enable notebook access). Kaggle: *Add-ons → Secrets*.
3. *Run all.* Community Forensics takes about **45 min** on a T4. The log prints *in-batch robust acc* every
   50 steps: it should climb well above 1% within the first epoch. If it stays near 0%, stop and tell Claude.
4. The EfficientNet step (2–3 h) runs **only if** Community Forensics reaches ≥ 30% robust accuracy at 4/255;
   otherwise the notebook skips it so you don't spend GPU time on a recipe that isn't working.

**What leaves this machine:** only weights and a JSON of numbers, uploaded to your **private** HF repo
`<your-username>/genned-robust` as `*-robust-v2.pt` (attempt 1's files stay for reference).

In [ ]:
!nvidia-smi || echo 'No GPU: switch the runtime to a GPU before continuing.'

In [ ]:
import os, subprocess

def secret(name):
    try:
        from google.colab import userdata  # Colab
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "Add an HF_TOKEN secret (write scope) first - see the first cell."
gh = secret("GITHUB_TOKEN")  # only needed if the repo is private
url = f"https://{gh}@github.com/as791/genned.git" if gh else "https://github.com/as791/genned.git"
if os.path.isdir("genned"):
    subprocess.run(["git", "-C", "genned", "pull", "--ff-only"], check=True)  # pick up attempt-2 code
else:
    subprocess.run(["git", "clone", "--depth", "1", url, "genned"], check=True)
os.chdir("genned")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "timm>=1.0,<2" "huggingface_hub>=0.23,<1" "datasets>=2.19,<5" "onnxruntime>=1.18,<2" "pillow>=10,<12"
from huggingface_hub import HfApi
HF_REPO = HfApi().whoami()["name"] + "/genned-robust"
print("Weights will be uploaded (privately) to", HF_REPO)

## Training data: Defactify train split only; validation: Defactify validation split
`--strict-split` makes the download fail rather than silently fall back to the test split the benchmarks use.
If you ran attempt 1 in this session, the train folder is reused.

In [ ]:
PER_CLASS = 3000  # images per class; lower it if the session is short on time or disk

!python tools/fetch_eval_data.py --dataset Rajarshi-Roy-research/Defactify_Image_Dataset \
    --split train --strict-split --out data/train/defactify --per-class {PER_CLASS} --max-scan 25000 \
    --label-col Label_A --ai-values 1 --real-values 0 \
    --generator-col Label_B --generator-names real,sd21,sdxl,sd3,dalle3,midjourney6
!python tools/fetch_eval_data.py --dataset Rajarshi-Roy-research/Defactify_Image_Dataset \
    --split validation --strict-split --out data/val/defactify --per-class 150 \
    --label-col Label_A --ai-values 1 --real-values 0 \
    --generator-col Label_B --generator-names real,sd21,sdxl,sd3,dalle3,midjourney6

## 1. Community Forensics ViT-S 224 (TRADES, 8 epochs, ~45 min on a T4)

In [ ]:
!python tools/adv_finetune.py --model commfor --objective trades \
    --train-data data/train/defactify --val-data data/val/defactify \
    --per-class {PER_CLASS} --tag v2 --out runs --push-to-hub {HF_REPO}

## 2. Dafilab EfficientNet-B4 (only if step 1 worked; 2–3 h on a T4)
Skipped automatically unless Community Forensics reached ≥ 30% robust accuracy at 4/255. CI can build the ensemble
with only the Community Forensics model fine-tuned.

In [ ]:
import json
log = json.load(open("runs/commfor-train-log-v2.json"))
robust = log["selected"]["robust_acc_4"]
print(f"Community Forensics: robust accuracy at 4/255 = {robust:.1%} (as shipped: {log['history'][0]['robust_acc_4']:.1%})")
if robust >= 0.30:
    !python tools/adv_finetune.py --model bundled --objective trades \
        --train-data data/train/defactify --val-data data/val/defactify \
        --per-class {PER_CLASS} --tag v2 --out runs --push-to-hub {HF_REPO}
else:
    print("Below 30%: skipping EfficientNet-B4. Tell Claude the numbers below.")

In [ ]:
import glob
for path in sorted(glob.glob("runs/*-train-log-v2.json")):
    log = json.load(open(path))
    print(f"{log['model']}: selected epoch {log['selected_epoch']}, clean gate met: {log['meets_clean_gate']}")
    for h in log["history"]:
        print(f"  epoch {h['epoch']}: clean AUC {h['clean_auc']:.3f}, "
              f"robust acc @2/255 {h['robust_acc_2']:.1%}, @4/255 {h['robust_acc_4']:.1%}")
print()
print("Done. Tell Claude:", HF_REPO, "(files *-robust-v2.pt)")